# Homework 6: Regression Model for Predicting Car Fuel Efficiency

* Dataset: https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv

In [1]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestRegressor

In [2]:
data = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv'

In [3]:
!wget $data

--2026-11-02 10:17:23--  https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 635180 (620K) [text/plain]
Saving to: ‘car_fuel_efficiency_2026.csv.1’

car_fuel_efficiency 100%[===================>] 620.29K  --.-KB/s    in 0.1s    

2026-11-02 10:17:24 (5.41 MB/s) - ‘car_fuel_efficiency_2026.csv.1’ saved [635180/635180]



In [4]:
!head car_fuel_efficiency_2026.csv

model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243,3870,,31.9
2008,Europe,Diesel,Front-wheel drive,4,2390,6,272,4210,,31.3
1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267,4240,17.3,27.5
1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258,4490,18.7,28.5
1994,USA,Diesel,Front-wheel drive,3,2580,7,304,4510,17.5,31.0
2007,USA,Hybrid,Front-wheel drive,2,2290,6,266,4260,18.8,31.2
1992,Asia,Gasoline,Front-wheel drive,2,2270,6,250,4310,21.7,26.8
2005,USA,Gasoline,Front-wheel drive,4,2370,6,279,4090,17.4,30.2
1997,Asia,Gasoline,Rear-wheel drive,3,2330,6,251,4230,19.5,30.9


## Data Preparation

In [5]:
df = pd.read_csv(data)

In [6]:
df.dtypes

model_year               int64
origin                     str
fuel_type                  str
drivetrain                 str
num_doors                int64
engine_displacement      int64
num_cylinders            int64
horsepower             float64
vehicle_weight           int64
acceleration           float64
fuel_efficiency_mpg    float64
dtype: object

In [7]:
df.columns = df.columns.str.lower() # Covnert to lowercase column names for consistency
df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [8]:
# This safely targets both older 'object' text and newer 'string' text
categorical_columns = list(df.select_dtypes(include=['object', 'string']).columns)

for c in categorical_columns:
    df[c] = df[c].str.lower().str.replace(' ', '_') # Standardizes object column data.
df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,europe,gasoline,front-wheel_drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,europe,diesel,front-wheel_drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,asia,gasoline,front-wheel_drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,europe,gasoline,front-wheel_drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,usa,diesel,front-wheel_drive,3,2580,7,304.0,4510,17.5,31.0


In [9]:
# Check missing values.
df.isnull().sum()

model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64

In [10]:
# Replace missing values from numerical features with 0.0
numerical_cols = list(df.select_dtypes(include=['int', 'float']).columns)
df[numerical_cols] = df[numerical_cols].fillna(0.0)
df.isnull().sum()

model_year             0
origin                 0
fuel_type              0
drivetrain             0
num_doors              0
engine_displacement    0
num_cylinders          0
horsepower             0
vehicle_weight         0
acceleration           0
fuel_efficiency_mpg    0
dtype: int64

In [11]:
df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,europe,gasoline,front-wheel_drive,4,2180,6,243.0,3870,0.0,31.9
1,2008,europe,diesel,front-wheel_drive,4,2390,6,272.0,4210,0.0,31.3
2,1996,asia,gasoline,front-wheel_drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,europe,gasoline,front-wheel_drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,usa,diesel,front-wheel_drive,3,2580,7,304.0,4510,17.5,31.0


In [12]:
# split the dataset
target = "fuel_efficiency_mpg"

df_full_train, df_test = train_test_split(
    df,
    test_size=0.2,
    random_state=1
)

df_train, df_val = train_test_split(
    df_full_train,
    test_size=0.25,
    random_state=1
)

y_train = df_train[target].values
y_val = df_val[target].values
y_test = df_test[target].values

del df_train[target]
del df_val[target]
del df_test[target]

In [13]:
# Transform the feature dataframes into sparse matrices

train_dicts = df_train.to_dict(orient="records")
val_dicts = df_val.to_dict(orient="records")

dv = DictVectorizer(sparse=True)

X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

In [14]:
# Define RMSE function

def rmse(y, y_pred):
    return np.sqrt(mean_squared_error(y, y_pred))

## Question 1

Let's train a decision tree regressor to predict the fuel_efficiency_mpg variable.

- Train a model with max_depth=1.

Which original feature is used for splitting the data? If you use DictVectorizer, map a one-hot feature such as origin=USA back to its original column name.

- 'vehicle_weight'
- 'model_year'
- 'origin'
- 'fuel_type'

In [15]:
# Train a decision tree regressor with `max_depth=1`

dt = DecisionTreeRegressor(
    max_depth=1,
    random_state=1
)

dt.fit(X_train, y_train)

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.For an example of how ``max_depth`` influences the model, see:ref:`sphx_glr_auto_examples_tree_plot_tree_regression.py`.",1
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",1
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 0.24 Poisson deviance criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split among considered features for this split.",'best'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: splitting may inspect more than ``max_features`` features ifneeded to find a valid split.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``m

In [16]:
# Inspect the trained tree to identify the feature used for the first split

feature_names = dv.get_feature_names_out()

print(
    export_text(
        dt,
        feature_names=list(feature_names)
    )
)

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]



## Question 2

Train a random forest regressor with these parameters:

- n_estimators=10
- random_state=1
- n_jobs=-1 (optional - to make training faster)

What's the RMSE of this model on the validation data?

- 0.1837
- 1.837
- 18.37
- 183.7

In [17]:
# Train a random forest regressor

rf = RandomForestRegressor(
    n_estimators=10,
    random_state=1,
    n_jobs=-1
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_val)

score = rmse(y_val, y_pred)

score

np.float64(1.8332197358745623)

In [18]:
print("Dataset shape:", df.shape)
print("Training shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Target mean:", y_train.mean())
print("Target standard deviation:", y_train.std())
print("RMSE:", rmse(y_val, rf.predict(X_val)))

Dataset shape: (10000, 11)
Training shape: (6000, 16)
Validation shape: (2000, 16)
Target mean: 30.01151666666667
Target standard deviation: 2.9306601701759205
RMSE: 1.8332197358745623


## Question 3

Now let's experiment with the n_estimators parameter

- Try these values of this parameter: [10, 50, 100, 150].
- Set random_state to 1.
- Evaluate the model on the validation dataset.

Which value gives the lowest validation RMSE? Consider 3 decimal places for calculating the answer.

- 10
- 50
- 100
- 150

In [19]:
# Train random forest models with different numbers of trees and compared validation RMSE

scores = {}

for n_estimators in [10, 50, 100, 150]:
    rf = RandomForestRegressor(
        n_estimators=n_estimators,
        random_state=1,
        n_jobs=-1
    )

    rf.fit(X_train, y_train)

    y_pred = rf.predict(X_val)

    score = rmse(y_val, y_pred)

    scores[n_estimators] = score

    print(n_estimators, round(score, 3))

10 1.833
50 1.774
100 1.77
150 1.769


## Question 4

Let's select the best max_depth:

- Try different values of max_depth: [10, 15, 20, 25]
- For each of these values,
  - try different values of n_estimators from [10, 50, 100, 150]
  - calculate the mean RMSE
- Fix the random seed: random_state=1

What's the best max_depth, using the mean RMSE?

- 10
- 15
- 20
- 25

In [20]:
# Test each combination of `max_depth` and `n_estimators`
# Calculate the mean RMSE across the `n_estimators` values

all_scores = []

for max_depth in [10, 15, 20, 25]:
    for n_estimators in [10, 50, 100, 150]:
        rf = RandomForestRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            random_state=1,
            n_jobs=-1
        )

        rf.fit(X_train, y_train)

        y_pred = rf.predict(X_val)

        score = rmse(y_val, y_pred)

        all_scores.append(
            {
                "max_depth": max_depth,
                "n_estimators": n_estimators,
                "rmse": score,
            }
        )

df_scores = pd.DataFrame(all_scores)

df_scores

,max_depth,n_estimators,rmse
0,10,10,1.782348
1,10,50,1.747520
2,10,100,1.743986
3,10,150,1.743716
4,15,10,1.831390
5,15,50,1.770883
6,15,100,1.767872
7,15,150,1.766752
8,20,10,1.837314
9,20,50,1.772377


In [21]:
# Calculate the mean RMSE for each value of `max_depth`

mean_scores = (
    df_scores
    .groupby("max_depth")["rmse"]
    .mean()
    .sort_values()
)

mean_scores

max_depth
10    1.754392
15    1.784224
25    1.786305
20    1.786771
Name: rmse, dtype: float64

## Question 5

We can extract feature importance information from tree-based models.

At each step of the decision tree learning algorithm, it finds the best split. When doing it, we can calculate "gain" - the reduction in impurity before and after the split. This gain is quite useful in understanding what are the important features for tree-based models.

In Scikit-Learn, tree-based models contain this information in the feature_importances_ field.

For this homework question, we'll find the most important feature:

- Train the model with these parameters:
  - n_estimators=10,
  - max_depth=20,
  - random_state=1,
  - n_jobs=-1 (optional)
- Get the feature importance information from this model

What's the most important feature (among these 4)?

- vehicle_weight
- horsepower
- acceleration
- engine_displacement

In [22]:
# Train the model specified
rf = RandomForestRegressor(
    n_estimators=10,
    max_depth=20,
    random_state=1,
    n_jobs=-1
)

rf.fit(X_train, y_train)

# Extract and sort feature importances
feature_importance = pd.DataFrame({
    "feature": dv.get_feature_names_out(),
    "importance": rf.feature_importances_
}).sort_values(
    by="importance",
    ascending=False
)

feature_importance.sort_values('importance', ascending=False).head()


,feature,importance
9,model_year,0.296107
15,vehicle_weight,0.203313
6,fuel_type=gasoline,0.160712
8,horsepower,0.078271
4,engine_displacement,0.061281


## Question 6

Now let's train an XGBoost model! For this question, we'll tune the eta parameter:

- Install XGBoost
- Create DMatrix for train and validation
- Create a watchlist
- Train a model with these parameters for 100 rounds:

```
xgb_params = {
    'eta': 0.3, 
    'max_depth': 6,
    'min_child_weight': 1,
    
    'objective': 'reg:squarederror',
    'nthread': 8,
    
    'seed': 1,
    'verbosity': 1,
}
```

Now change eta from 0.3 to 0.1.

Which eta leads to the best RMSE score on the validation dataset?

- 0.3
- 0.1
- Both give equal value

In [62]:
# Install XGBoost 
!uv add xgboost

Resolved 118 packages in 595ms                                       
Installed 2 packages in 16ms2.3                                  
 + nvidia-nccl-cu13==2.32.3
 + xgboost==3.4.1


In [63]:
# Create XGBoost `DMatrix` objects for the training and validation data

import xgboost as xgb

features = dv.get_feature_names_out().tolist()

dtrain = xgb.DMatrix(
    X_train,
    label=y_train,
    feature_names=features
)

dval = xgb.DMatrix(
    X_val,
    label=y_val,
    feature_names=features
)

watchlist = [
    (dtrain, "train"),
    (dval, "val"),
]

In [64]:
# Model with `eta = 0.3`

xgb_params = {
    "eta": 0.3,
    "max_depth": 6,
    "min_child_weight": 1,
    "objective": "reg:squarederror",
    "nthread": 8,
    "seed": 1,
    "verbosity": 1,
}

model_eta_03 = xgb.train(
    params=xgb_params,
    dtrain=dtrain,
    num_boost_round=100,
    evals=watchlist
)

y_pred_eta_03 = model_eta_03.predict(dval)

rmse_eta_03 = rmse(y_val, y_pred_eta_03)

rmse_eta_03

[0]	train-rmse:2.40653	val-rmse:2.46782
[1]	train-rmse:2.07098	val-rmse:2.15784
[2]	train-rmse:1.86454	val-rmse:1.97398
[3]	train-rmse:1.73360	val-rmse:1.86662
[4]	train-rmse:1.65059	val-rmse:1.80312
[5]	train-rmse:1.59328	val-rmse:1.76216
[6]	train-rmse:1.55435	val-rmse:1.74392
[7]	train-rmse:1.52677	val-rmse:1.73976
[8]	train-rmse:1.50265	val-rmse:1.73478
[9]	train-rmse:1.48761	val-rmse:1.73282
[10]	train-rmse:1.47125	val-rmse:1.73205
[11]	train-rmse:1.45902	val-rmse:1.73227
[12]	train-rmse:1.44829	val-rmse:1.73503
[13]	train-rmse:1.43604	val-rmse:1.73466
[14]	train-rmse:1.42972	val-rmse:1.73448
[15]	train-rmse:1.41908	val-rmse:1.73512
[16]	train-rmse:1.41206	val-rmse:1.73297
[17]	train-rmse:1.40940	val-rmse:1.73517
[18]	train-rmse:1.40278	val-rmse:1.73584
[19]	train-rmse:1.39637	val-rmse:1.73664
[20]	train-rmse:1.39368	val-rmse:1.73766
[21]	train-rmse:1.39254	val-rmse:1.73701
[22]	train-rmse:1.37639	val-rmse:1.74216
[23]	train-rmse:1.36470	val-rmse:1.74363
[24]	train-rmse:1.36071	va

np.float64(1.8264579799904583)

In [65]:
# Model with `eta = 0.1`

xgb_params = {
    "eta": 0.1,
    "max_depth": 6,
    "min_child_weight": 1,
    "objective": "reg:squarederror",
    "nthread": 8,
    "seed": 1,
    "verbosity": 1,
}

model_eta_01 = xgb.train(
    params=xgb_params,
    dtrain=dtrain,
    num_boost_round=100,
    evals=watchlist
)

y_pred_eta_01 = model_eta_01.predict(dval)

rmse_eta_01 = rmse(y_val, y_pred_eta_01)

rmse_eta_01

[0]	train-rmse:2.74739	val-rmse:2.78635
[1]	train-rmse:2.58527	val-rmse:2.63162
[2]	train-rmse:2.44540	val-rmse:2.50036
[3]	train-rmse:2.32187	val-rmse:2.38568
[4]	train-rmse:2.21445	val-rmse:2.28765
[5]	train-rmse:2.12109	val-rmse:2.20428
[6]	train-rmse:2.03996	val-rmse:2.13241
[7]	train-rmse:1.96795	val-rmse:2.06798
[8]	train-rmse:1.90640	val-rmse:2.01618
[9]	train-rmse:1.85240	val-rmse:1.96998
[10]	train-rmse:1.80592	val-rmse:1.93152
[11]	train-rmse:1.76410	val-rmse:1.89688
[12]	train-rmse:1.72781	val-rmse:1.86726
[13]	train-rmse:1.69564	val-rmse:1.84312
[14]	train-rmse:1.66745	val-rmse:1.82218
[15]	train-rmse:1.64286	val-rmse:1.80346
[16]	train-rmse:1.62187	val-rmse:1.79038
[17]	train-rmse:1.60321	val-rmse:1.77726
[18]	train-rmse:1.58694	val-rmse:1.76827
[19]	train-rmse:1.57240	val-rmse:1.75951
[20]	train-rmse:1.55847	val-rmse:1.75076
[21]	train-rmse:1.54640	val-rmse:1.74347
[22]	train-rmse:1.53643	val-rmse:1.73912
[23]	train-rmse:1.52675	val-rmse:1.73485
[24]	train-rmse:1.51763	va

np.float64(1.724810059827976)